# Experiment 1 · MuSiQue, 0 to 16 distractors, NARS vs 9 LLMs (Colab)

**Question:** as distractor paragraphs are added, does NARS lose less accuracy than LLMs answering directly, and how
does that compare across model sizes?

200 MuSiQue 2-hop questions (`questions/MuSiQue_exp1.pkl`, from `build_questions.ipynb`), each read with k = 0, 4, 8
and 16 distractor paragraphs. Every system reads the same 70B-extracted facts; the distractor sets nest (the 4 are
among the 8) and keep their reading positions. ONA.py and the LoRA adapter are frozen.

| System | Kind | Served by |
|---|---|---|
| NARS + 8B + LoRA | the distilled 8B writes triples and rules; ONA reasons | vLLM on this GPU |
| NARS + DeepSeek-V3 | the teacher writes triples and rules; ONA reasons | DeepInfra |
| Llama-3.1-8B | direct answer (the same weights as the student) | vLLM on this GPU |
| gemma-3-4b · gemma-3-12b · gemma-3-27b · phi-4 · Mistral-Small-24B · Llama-3.3-70B · DeepSeek-V3 · Qwen3-Next-80B-A3B | direct answer | DeepInfra |

Scoring: **exact** match, **lenient** match and the **judge** (Qwen3-235B, not one of the systems, strict prompt).
Main analysis: each system's accuracy change with distractors relative to Llama-3.1-8B (logistic regression with
questions as clusters, and the paired change from k = 0 to 16).

**Before you run:**
1. Runtime → Change runtime type → **A100**.
2. Colab secrets (🔑 on the left, *Notebook access* on): `DEEPINFRA_API_KEY`, and `HF_TOKEN` (a Hugging Face **read**
   token: the adapter repository is private).
3. Runtime → **Run all**. Setup about 10 minutes; the run about 6–8 hours (the NARS rows; the direct rows take
   minutes). DeepInfra about $13–15.

**Saved to Drive** (`MyDrive/nars_lora/experiment_1/`), as each answer finishes, so a stopped session resumes:
- `config.json`: models, prompts, versions, question-file hash (written once).
- `answers/k{k}__{system}.jsonl`: one line per answer (inputs read, raw reply, answer, tokens, cost, time, all metrics).
- `traces/k{k}__{system}.jsonl`: NARS rows only, everything NARS did (triples, plan, rules, votes, Narsese, ONA output,
  every LLM call).
- `results.csv`, `report.txt`, `accuracy_vs_k.png`: written by the Results cell; rerun it any time.

In [ ]:
%%capture
# The code (experiment_1 folder), ONA at the version used for every result, vLLM
!git clone -q https://github.com/MinaGabriel/llm-vs-reasoning.git /content/llm-vs-reasoning || git -C /content/llm-vs-reasoning pull -q
!git clone -q https://github.com/opennars/OpenNARS-for-Applications.git /content/ONA || true
!cd /content/ONA && git checkout -q dc4efd0abd520cdb79bf53bfa3c285ebb24f2e8a && sh build.sh
!pip install -q vllm prettytable statsmodels

In [ ]:
import os, subprocess, sys, time, urllib.request
from google.colab import drive, userdata
from huggingface_hub import HfApi, snapshot_download

os.chdir("/content/llm-vs-reasoning/experiment_1")
sys.path.insert(0, "/content/llm-vs-reasoning/experiment_1")
os.environ["DEEPINFRA_API_KEY"] = userdata.get("DEEPINFRA_API_KEY")
assert os.path.exists("/content/ONA/NAR"), "ONA did not build: run the first cell without %%capture to see why"
assert os.path.exists("questions/MuSiQue_exp1.pkl"), "questions/MuSiQue_exp1.pkl is missing from the repository"
drive.mount("/content/drive")

ADAPTER_REPO = "MinaGabriel/nars-llama31-8b-lora"
ADAPTER_REVISION = HfApi().model_info(ADAPTER_REPO, token=userdata.get("HF_TOKEN")).sha
ADAPTER = snapshot_download(ADAPTER_REPO, revision=ADAPTER_REVISION, token=userdata.get("HF_TOKEN"), local_dir="/content/adapter")
assert os.path.exists(f"{ADAPTER}/adapter_model.safetensors"), f"no adapter in {ADAPTER}"

# vLLM: the base 8B (Meta's weights, ungated copy: the one the adapter was trained on) plus the adapter
server = subprocess.Popen(
    ["vllm", "serve", "unsloth/Meta-Llama-3.1-8B-Instruct", "--served-model-name", "llama-3.1-8b-instruct",
     "--dtype", "bfloat16", "--enable-lora", "--lora-modules", f"nars-lora={ADAPTER}", "--max-lora-rank", "16",
     "--max-model-len", "16384", "--gpu-memory-utilization", "0.90", "--port", "8000"],
    stdout=open("/content/vllm.log", "w"), stderr=subprocess.STDOUT)
for minute in range(20):                          # the first start downloads 16 GB of weights
    time.sleep(30)
    try:
        urllib.request.urlopen("http://localhost:8000/health", timeout=5)
        print("vLLM is up")
        break
    except Exception:
        if server.poll() is not None:
            print(open("/content/vllm.log").read()[-3000:])
            raise SystemExit("vLLM stopped: see the log above")
        print(f"starting vLLM... {(minute + 1) / 2:.1f} min")
else:
    raise SystemExit("vLLM did not start in 10 minutes: see /content/vllm.log")

In [ ]:
# ── Settings (frozen for the experiment) ─────────────────────────────────────────────────────────
import hashlib, json, math, pickle, random, re, unicodedata
from concurrent.futures import ThreadPoolExecutor, as_completed
from datetime import datetime, timezone

from openai import OpenAI
from prettytable import PrettyTable

import ONA
from keys import deepinfra_key

CLIENTS = {
    "deepinfra": OpenAI(api_key=deepinfra_key(), base_url="https://api.deepinfra.com/v1/openai", max_retries=3, timeout=120),
    "vllm": OpenAI(api_key="none", base_url="http://localhost:8000/v1", max_retries=2, timeout=600),
}
SYSTEMS = {                                       # name → kind, server, model, size label (for tables and plots)
    "NARS + 8B + LoRA":   {"kind": "nars",   "server": "vllm",      "model": "nars-lora",                                   "size": "8B"},
    "NARS + DeepSeek-V3": {"kind": "nars",   "server": "deepinfra", "model": "deepseek-ai/DeepSeek-V3",                     "size": "671B MoE"},
    "Llama-3.1-8B":       {"kind": "direct", "server": "vllm",      "model": "llama-3.1-8b-instruct",                       "size": "8B"},
    "gemma-3-4b":         {"kind": "direct", "server": "deepinfra", "model": "google/gemma-3-4b-it",                        "size": "4B"},
    "gemma-3-12b":        {"kind": "direct", "server": "deepinfra", "model": "google/gemma-3-12b-it",                       "size": "12B"},
    "phi-4":              {"kind": "direct", "server": "deepinfra", "model": "microsoft/phi-4",                             "size": "14B"},
    "Mistral-Small-24B":  {"kind": "direct", "server": "deepinfra", "model": "mistralai/Mistral-Small-3.2-24B-Instruct-2506", "size": "24B"},
    "gemma-3-27b":        {"kind": "direct", "server": "deepinfra", "model": "google/gemma-3-27b-it",                       "size": "27B"},
    "Llama-3.3-70B":      {"kind": "direct", "server": "deepinfra", "model": "meta-llama/Llama-3.3-70B-Instruct-Turbo",     "size": "70B"},
    "Qwen3-Next-80B-A3B": {"kind": "direct", "server": "deepinfra", "model": "Qwen/Qwen3-Next-80B-A3B-Instruct",            "size": "80B MoE"},
    "DeepSeek-V3":        {"kind": "direct", "server": "deepinfra", "model": "deepseek-ai/DeepSeek-V3",                     "size": "671B MoE"},
}
REFERENCE = "Llama-3.1-8B"                        # the matched baseline: the student's own weights, answering directly
K_VALUES = [0, 4, 8, 16]
WORKERS = 24
ONA_OPTIONS = {"plan": True, "votes": 3, "merge_relations": True}
ONA_PATH = "/content/ONA/NAR"
ONA_COMMIT = "dc4efd0abd520cdb79bf53bfa3c285ebb24f2e8a"
JUDGE_MODEL = "Qwen/Qwen3-235B-A22B-Instruct-2507"   # not one of the systems
RUN_DIR = "/content/drive/MyDrive/nars_lora/experiment_1"
QUESTION_FILE = "questions/MuSiQue_exp1.pkl"
for folder in ("answers", "traces"):
    os.makedirs(f"{RUN_DIR}/{folder}", exist_ok=True)

QUESTIONS = pickle.load(open(QUESTION_FILE, "rb"))
QUESTION_HASH = hashlib.sha256(open(QUESTION_FILE, "rb").read()).hexdigest()
print(f"{len(QUESTIONS)} questions · k = {K_VALUES} · {len(SYSTEMS)} systems")


def load_prices():
    try:
        with urllib.request.urlopen("https://api.deepinfra.com/models/list", timeout=20) as response:
            return {m["model_name"]: (m["pricing"]["cents_per_input_token"] * 1e4, m["pricing"]["cents_per_output_token"] * 1e4)
                    for m in json.load(response) if (m.get("pricing") or {}).get("type") == "tokens"}
    except Exception:
        return {}


PRICES = load_prices()


def cost(input_tokens, output_tokens, model):     # vLLM rows cost GPU time only: $0 here
    price = PRICES.get(model, (0, 0))
    return (input_tokens * price[0] + output_tokens * price[1]) / 1_000_000


# Every model must answer under its own name (DeepInfra has silently served retired models with another one)
for spec in [*SYSTEMS.values(), {"server": "deepinfra", "model": JUDGE_MODEL}]:
    served = CLIENTS[spec["server"]].chat.completions.create(
        model=spec["model"], max_tokens=1, messages=[{"role": "user", "content": "Hi"}]).model
    assert served == spec["model"], f"{spec['model']} is served as {served}"
print("all models answer under their own names")

In [ ]:
# ── Inputs, prompts and scoring ──────────────────────────────────────────────────────────────────
def paragraphs_at(q, k):
    """The 2 supporting paragraphs and the first k distractors, in their fixed reading order (nested in k)."""
    return [p for p in q["layout"] if p["is_supporting"] or p["rank"] < k]


def facts(q, k):
    return [p["facts"] for p in paragraphs_at(q, k)]


def golds(q):
    return list(dict.fromkeys([q["answer"], *(q.get("gold_answers") or [])]))


def normalize(text):
    text = unicodedata.normalize("NFKD", str(text)).encode("ascii", "ignore").decode().casefold()
    return re.sub(r"[^a-z0-9]", "", re.sub(r"\b(a|an|the)\b", " ", text))


def exact(answer, g):
    return any(normalize(answer or "") == normalize(x) for x in g)


def lenient(answer, g):
    """One contains the other, the shorter covering at least half of the longer."""
    a = normalize(answer or "")
    for x in map(normalize, g):
        short, long_ = sorted((a, x), key=len)
        if short and short in long_ and 2 * len(short) >= len(long_):
            return True
    return False


DIRECT_PROMPT = """Answer the question using only the facts below. Return only the short answer, without explanation.

Facts:
{facts}

Question: {question}
Answer:"""

JUDGE_PROMPT = """You grade one answer to a question against the gold answer. Be strict and consistent.

Question: {question}
Gold answer: {gold}
Model answer: {answer}

YES if the model answer gives the same entity, date, number or place as the gold answer, even with extra words around
it, a fuller or shorter name, or an abbreviation:
  "IOC president Thomas Bach" for "Thomas Bach"; "UN (United Nations)" for "United Nations";
  "Marie Curie won it" for "Marie Curie"; "the 1990s" for "1990s"
NO if it gives anything else, even when it shares words with the gold answer or is related to it:
  a different thing with a similar name ("The Boston Globe" for "Boston"); a part or a larger whole ("Oregon" for
  "Portland"; "Portland" for "Oregon"); something about the answer instead of it ("Einstein's theory" for "Einstein");
  several candidates ("Paris or Lyon" for "Paris"); a different date or number; no answer ("not stated").
Reply with only yes or no."""


def metrics(q, answer):
    """exact; lenient; judge = exact, or else the judge model says it is the same thing (its raw reply is kept)."""
    g = golds(q)
    result = {"exact": exact(answer, g), "lenient": lenient(answer, g), "judge_reply": None, "judge_cost": 0.0}
    result["judge"] = result["exact"]
    if answer and not result["exact"]:
        response = CLIENTS["deepinfra"].chat.completions.create(model=JUDGE_MODEL, temperature=0, max_tokens=3, messages=[{
            "role": "user", "content": JUDGE_PROMPT.format(question=q["question"], gold=" / ".join(g), answer=answer)}])
        result["judge_reply"] = (response.choices[0].message.content or "").strip()
        result["judge"] = result["judge_reply"].lower().startswith("yes")
        result["judge_cost"] = cost(response.usage.prompt_tokens or 0, response.usage.completion_tokens or 0, JUDGE_MODEL)
    return result


def write_config():
    """Everything needed to say exactly what was run (written once; a mismatch on resume stops the run)."""
    config = {"experiment": "experiment_1", "dataset": "MuSiQue", "questions": len(QUESTIONS), "question_file": QUESTION_FILE,
              "question_sha256": QUESTION_HASH, "k_values": K_VALUES, "systems": SYSTEMS, "reference": REFERENCE,
              "ona_options": ONA_OPTIONS, "ona_commit": ONA_COMMIT, "ona_py_sha256": hashlib.sha256(open("ONA.py", "rb").read()).hexdigest(),
              "adapter": {"repo": ADAPTER_REPO, "revision": ADAPTER_REVISION}, "base_model": "unsloth/Meta-Llama-3.1-8B-Instruct (bf16, vLLM)",
              "judge": JUDGE_MODEL, "prompts": {"direct": DIRECT_PROMPT, "judge": JUDGE_PROMPT},
              "direct_settings": {"temperature": 0, "max_tokens": 64, "answer": "first line of the reply"},
              "repository_commit": subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip(),
              "created": datetime.now(timezone.utc).isoformat()}
    path = f"{RUN_DIR}/config.json"
    if os.path.exists(path):
        old = json.load(open(path))
        for key in ("question_sha256", "ona_py_sha256", "adapter", "systems", "k_values", "prompts", "judge"):
            assert old[key] == json.loads(json.dumps(config[key])), f"config changed since the first run: {key}"
        print("config.json matches the first run: resuming")
    else:
        json.dump(config, open(path, "w"), indent=2)
        print("wrote config.json")


write_config()

In [ ]:
# ── One answer ───────────────────────────────────────────────────────────────────────────────────
def run_one(index, system, k):
    """Returns (answer record, trace or None). Errors are saved and retried on the next pass."""
    q, spec = QUESTIONS[index], SYSTEMS[system]
    client, model = CLIENTS[spec["server"]], spec["model"]
    read = paragraphs_at(q, k)
    record = {"index": index, "id": q["id"], "source": q["source_split"], "question": q["question"], "gold": q["answer"],
              "golds": golds(q), "k": k, "system": system, "kind": spec["kind"], "model": model, "size": spec["size"],
              "paragraphs": [{"title": p["title"], "supporting": p["is_supporting"], "rank": p["rank"]} for p in read],
              "facts_read": sum(len(p["facts"]) for p in read), "error": None, "cost": 0.0,
              "started": datetime.now(timezone.utc).isoformat()}
    trace, start = None, time.time()
    try:
        if spec["kind"] == "direct":
            prompt = DIRECT_PROMPT.format(facts="\n".join(f"- {f}" for p in facts(q, k) for f in p), question=q["question"])
            response = client.chat.completions.create(model=model, temperature=0, max_tokens=64,
                                                      messages=[{"role": "user", "content": prompt}])
            raw = response.choices[0].message.content or ""
            record.update(served_model=response.model, raw_reply=raw, answer=raw.strip().split("\n")[0] or None,
                          finish_reason=response.choices[0].finish_reason, prompt_sha256=hashlib.sha256(prompt.encode()).hexdigest(),
                          input_tokens=response.usage.prompt_tokens, output_tokens=response.usage.completion_tokens,
                          cost=cost(response.usage.prompt_tokens or 0, response.usage.completion_tokens or 0, model))
        else:
            ona = ONA.OnaReasoner(client, model=model, **ONA_OPTIONS)
            answer = ona.solve(facts(q, k), q["question"], ONA_PATH)
            record.update(answer=answer, has_rules=bool(ona.last_rules), rules=ona.last_rules, votes=[s["answer"] for s in ona.samples],
                          auto_repairs=dict(ona.auto_repairs), answer_type=ona.answer_type, ona_top=ona.ona_top,
                          triples=len(ona.last_triples), llm_calls=len(ona.request_log),
                          input_tokens=ona.usage["input_tokens"], output_tokens=ona.usage["output_tokens"],
                          cost=cost(ona.usage["input_tokens"], ona.usage["output_tokens"], model))
            trace = {"index": index, "id": q["id"], "k": k, "system": system, "answer": answer, "plan": ona.plan_text,
                     "triples": ona.last_triples, "aliases": ona.aliases, "relation_groups": ona.relation_groups,
                     "missing_terms": ona.missing_terms, "rule_attempts": ona.rule_attempts, "rules": ona.last_rules,
                     "samples": [{key: s.get(key) for key in ("answer", "rules", "auto_repairs")} for s in ona.samples],
                     "candidates": [ona.names.get(c, c) for c in ona.candidates], "narsese": ona.last_narsese,
                     "ona_output": ona.last_output, "llm_calls": ona.request_log}
        record.update(metrics(q, record["answer"]))
    except Exception as error:
        record["error"] = f"{type(error).__name__}: {str(error)[:300]}"
    record["seconds"] = round(time.time() - start, 1)
    return record, trace


def stem(system, k):
    return f"k{k:02d}__{re.sub(r'[^A-Za-z0-9]+', '_', system).strip('_')}"


def load(system, k):
    path = f"{RUN_DIR}/answers/{stem(system, k)}.jsonl"
    if not os.path.exists(path):
        return {}
    with open(path) as f:
        return {r["index"]: r for r in map(json.loads, f) if not r.get("error")}

In [ ]:
# ── Run everything that is missing (direct rows first: they take minutes; up to 3 passes) ────────
spent, start = 0.0, time.time()
order = sorted(SYSTEMS, key=lambda s: SYSTEMS[s]["kind"] == "nars")
for attempt in range(3):
    jobs = [(i, s, k) for s in order for k in K_VALUES for i in range(len(QUESTIONS)) if i not in load(s, k)]
    total = len(QUESTIONS) * len(SYSTEMS) * len(K_VALUES)
    print(f"pass {attempt + 1}: {len(jobs)} answers to run ({total - len(jobs)} of {total} saved)")
    if not jobs:
        break
    with ThreadPoolExecutor(WORKERS) as executor:
        futures = {executor.submit(run_one, i, s, k): (s, k) for i, s, k in jobs}
        for done, future in enumerate(as_completed(futures), 1):
            system, k = futures[future]
            record, trace = future.result()
            with open(f"{RUN_DIR}/answers/{stem(system, k)}.jsonl", "a") as f:
                f.write(json.dumps(record, ensure_ascii=False, default=str) + "\n")
            if trace is not None:
                with open(f"{RUN_DIR}/traces/{stem(system, k)}.jsonl", "a") as f:
                    f.write(json.dumps(trace, ensure_ascii=False, default=str) + "\n")
            spent += record["cost"] + record.get("judge_cost", 0.0)
            mark = "!" if record["error"] else "✓" if record["judge"] else "✗"
            if record["kind"] == "nars" or record["error"] or done % 50 == 0:
                print(f"[{done:>5}/{len(jobs)}] k={k:<2} {system:<18} {mark} {str(record.get('answer') or '-')[:26]:<26} "
                      f"gold {record['gold'][:22]:<22} ${spent:.2f} {(time.time() - start) / 60:.0f} min")
missing = sum(1 for s in SYSTEMS for k in K_VALUES for i in range(len(QUESTIONS)) if i not in load(s, k))
print(f"\ndone: ${spent:.2f} this session" + (f" · {missing} answers still failing: run this cell again" if missing else ""))

## Results

Reads everything saved on Drive, so it can be run any time (also while answers are still missing).

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf

rows = [r for s in SYSTEMS for k in K_VALUES for r in load(s, k).values()]
df = pd.DataFrame(rows)
df.drop(columns=["paragraphs", "rules", "votes", "auto_repairs", "golds"], errors="ignore").to_csv(f"{RUN_DIR}/results.csv", index=False)
for metric in ("exact", "lenient", "judge"):
    df[metric] = df[metric].astype(int)
names = [s for s in SYSTEMS if s in set(df.system)]
lines = [f"EXPERIMENT 1 · MuSiQue · {df['index'].nunique()} questions · {len(df)} answers", ""]


def mcnemar(a, b):
    n, m = a + b, min(a, b)
    return 1.0 if n == 0 else min(1.0, 2 * sum(math.comb(n, i) for i in range(m + 1)) / 2 ** n)


for metric in ("judge", "exact"):
    table = PrettyTable(["System", "size", *[f"k = {k}" for k in K_VALUES], "change 0 → 16"])
    table.align["System"] = "l"
    for s in names:
        acc = [df[(df.system == s) & (df.k == k)][metric].mean() * 100 for k in K_VALUES]
        table.add_row([s, SYSTEMS[s]["size"], *[f"{a:.1f}%" for a in acc], f"{acc[-1] - acc[0]:+.1f}"])
    lines += [f"ACCURACY · {metric}", str(table), ""]


def drop_gap(metric, a, b=REFERENCE, draws=5000):
    """(change of A from k=0 to 16) - (change of B), per question, with a bootstrap 95% CI over questions."""
    w = df[df.system.isin([a, b]) & df.k.isin([0, 16])].pivot_table(index="index", columns=["system", "k"], values=metric).dropna()
    diff = ((w[a][16] - w[a][0]) - (w[b][16] - w[b][0])).to_numpy()
    rng = random.Random(0)
    boots = sorted(sum(rng.choice(diff) for _ in diff) / len(diff) for _ in range(draws))
    return 100 * diff.mean(), 100 * boots[int(0.025 * draws)], 100 * boots[int(0.975 * draws)], len(diff)


table = PrettyTable(["System", "judge: less lost than " + REFERENCE + " (k 0 → 16)", "exact"])
table.align["System"] = "l"
for s in names:
    if s != REFERENCE:
        cells = []
        for metric in ("judge", "exact"):
            mean, lo, hi, n = drop_gap(metric, s)
            cells.append(f"{mean:+.1f} points [{lo:+.1f}, {hi:+.1f}]")
        table.add_row([s, *cells])
lines += ["ROBUSTNESS vs " + REFERENCE + ": per question, 95% bootstrap CI (positive = loses less)", str(table), ""]

df["dose"] = (df.k + 1).apply(math.log2)
table = PrettyTable(["System", "judge: slope vs " + REFERENCE, "p", "exact: slope", "p "])
table.align["System"] = "l"
fits = {m: smf.gee(f"{m} ~ C(system, Treatment('{REFERENCE}')) * dose", groups="index", data=df,
                   family=sm.families.Binomial()).fit() for m in ("judge", "exact")}
for s in names:
    if s != REFERENCE:
        term = f"C(system, Treatment('{REFERENCE}'))[T.{s}]:dose"
        table.add_row([s, *[x for m in ("judge", "exact") for x in (f"{fits[m].params[term]:+.3f}", f"{fits[m].pvalues[term]:.4f}")]])
lines += ["INTERACTION: logistic regression correct ~ system × log2(1 + k), questions as clusters (GEE);",
          "slope = extra log-odds per doubling of (1 + k) relative to " + REFERENCE, str(table), ""]

wide = df.pivot_table(index=["index", "k"], columns="system", values="judge").reset_index()
table = PrettyTable(["judge, same questions", *[f"k = {k}" for k in K_VALUES]])
table.align["judge, same questions"] = "l"
for a in [s for s in names if SYSTEMS[s]["kind"] == "nars"]:
    for b in [REFERENCE, "DeepSeek-V3", "Llama-3.3-70B"]:
        if b in names:
            cells = []
            for k in K_VALUES:
                w = wide[wide.k == k].dropna(subset=[a, b])
                x, y = int(((w[a] == 1) & (w[b] == 0)).sum()), int(((w[b] == 1) & (w[a] == 0)).sum())
                cells.append(f"+{x} / -{y}  p={mcnemar(x, y):.3f}")
            table.add_row([f"{a} vs {b}", *cells])
lines += ["PAIRED: +only A right / -only B right, exact McNemar p", str(table), ""]

nars = df[df.kind == "nars"]
table = PrettyTable(["NARS system", *[f"no rules k={k}" for k in K_VALUES], *[f"no answer k={k}" for k in K_VALUES]])
for s in sorted(set(nars.system)):
    part = nars[nars.system == s]
    table.add_row([s, *[f"{(~part[part.k == k].has_rules.astype(bool)).mean() * 100:.0f}%" for k in K_VALUES],
                   *[f"{part[part.k == k].answer.isna().mean() * 100:.0f}%" for k in K_VALUES]])
lines += ["NARS FAILURES: no valid rule / no answer derived", str(table), ""]

lines.append("BY SOURCE (judge accuracy at k = " + " / ".join(map(str, K_VALUES)) + ")")
for source, part in df.groupby("source"):
    lines.append(f"   {source} (n={part['index'].nunique()})")
    for s in names:
        lines.append(f"      {s:<20} " + " / ".join(f"{part[(part.system == s) & (part.k == k)].judge.mean() * 100:.0f}" for k in K_VALUES))
lines.append(f"\nCOST: ${df.cost.sum() + df.judge_cost.sum():.2f} on DeepInfra (systems ${df.cost.sum():.2f}, judge ${df.judge_cost.sum():.2f})")

report = "\n".join(lines)
print(report)
with open(f"{RUN_DIR}/report.txt", "w") as f:
    f.write(report + "\n")

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5), sharey=True)
for ax, metric in zip(axes, ("judge", "exact")):
    for s in names:
        acc, err = [], []
        for k in K_VALUES:
            v = df[(df.system == s) & (df.k == k)][metric]
            p, n = v.mean(), len(v)
            acc.append(100 * p)
            err.append(196 * math.sqrt(p * (1 - p) / max(n, 1)))
        is_nars = SYSTEMS[s]["kind"] == "nars"
        ax.errorbar(K_VALUES, acc, yerr=err if is_nars or s == REFERENCE else None, marker="o", capsize=3,
                    linewidth=2.5 if is_nars else 1.2, linestyle="-" if is_nars or s == REFERENCE else "--", label=s)
    ax.set(title=f"MuSiQue · {metric}", xlabel="distractor paragraphs (k)", xticks=K_VALUES)
    ax.grid(alpha=0.3)
axes[0].set_ylabel("accuracy (%) · 95% CI for NARS and " + REFERENCE)
axes[1].legend(loc="center left", bbox_to_anchor=(1.01, 0.5), fontsize=9)
plt.tight_layout()
plt.savefig(f"{RUN_DIR}/accuracy_vs_k.png", dpi=200, bbox_inches="tight")
plt.show()
print("saved results.csv, report.txt and accuracy_vs_k.png to", RUN_DIR)

## Done: release the GPU

In [ ]:
from google.colab import runtime
server.terminate()
drive.flush_and_unmount()
runtime.unassign()